# Fixed-scenario evaluation collection

In [41]:
# Run notebook commands from the project root.
%cd /workspaces/tesis_v4

from pathlib import Path
import json
import torch

ROOT = Path.cwd()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_DIR = ROOT / 'data/eval'
MANIFEST = EVAL_DIR / 'seeds.json'
OPERATOR = 'Malcom'  # Replace before collecting.
print(f'Root: {ROOT} | Device: {DEVICE}')
print(f'Manifest: {MANIFEST}')
print(f'Operator: {OPERATOR}')

/workspaces/tesis_v4
Root: /workspaces/tesis_v4 | Device: cuda
Manifest: /workspaces/tesis_v4/data/eval/seeds.json
Operator: Malcom


## Protocol

In [34]:
# One primary trajectory per seed gives track diversity; target 700–1000 steps and keep clean partial runs >=500.
# First collect dev seeds 100–104; use val only for model selection and never inspect locked_test outputs during development.
# Drive naturally; on selected dev runs add one short coast and one conservative recovery, never repeated artificial failures.
COLLECTION_PLAN = [
    ('dev pilot', '100–104', 5, 'UX, labels, diagnostics'),
    ('dev complete', '100–107', 8, 'implementation checks'),
    ('validation', '200–207', 8, 'architecture selection'),
    ('locked test', '300–307', 8, 'final thesis only'),
]
for phase, seeds, episodes, use in COLLECTION_PLAN:
    print(f'{phase:14} | {seeds:9} | {episodes} trajectories | {use}')

# Allowed observed tags: competent_drive, curve_left, curve_right, coast, recovery, off_road, low_speed.

dev pilot      | 100–104   | 5 trajectories | UX, labels, diagnostics
dev complete   | 100–107   | 8 trajectories | implementation checks
validation     | 200–207   | 8 trajectories | architecture selection
locked test    | 300–307   | 8 trajectories | final thesis only


## Freeze seed splits

In [5]:
# Run once before collection; it intentionally refuses to overwrite an existing manifest.
!rwm eval \
	init-seeds data/eval/seeds.json \
	--dev-seeds 100,101,102,103,104,105,106,107 \
	--val-seeds 200,201,202,203,204,205,206,207 \
	--test-seeds 300,301,302,303,304,305,306,307

Seed manifest saved to data/eval/seeds.json
  Dev: 100,101,102,103,104,105,106,107
  Val: 200,201,202,203,204,205,206,207
  Test: 300,301,302,303,304,305,306,307


## Collect one development trajectory

In [16]:
# Change only the registered dev seed; Escape or closing the window saves the partial episode.
!SDL_AUDIODRIVER=dummy rwm eval \
	collect data/eval/seeds.json 104 \
	--policy-name human \
	--render-mode human \
	--fps 60 \
	--out-dir data/eval \
	--operator {OPERATOR} \
	--max-steps 800

Saved episode: data/eval/dev/dev_104_20260717_161800.npz


## Review and label

In [20]:
# Inspect the latest dev metadata before labelling; change tags to events actually present in this episode.
DEV_EPISODES = sorted((EVAL_DIR / 'dev').glob('*.npz'))
assert DEV_EPISODES, 'Collect a development episode first.'
LATEST_EPISODE = DEV_EPISODES[-2]
metadata_path = LATEST_EPISODE.with_suffix('.episode.json')
print(metadata_path)
#print(json.dumps(json.loads(metadata_path.read_text()), indent=2))

/workspaces/tesis_v4/data/eval/dev/dev_103_20260717_161728.episode.json


In [21]:
# Label only after review; use review instead of keep when unsure.
!rwm eval label {LATEST_EPISODE} \
	--quality keep \
	--tags competent_drive \
	--operator {OPERATOR} \
	--notes 'Centered driving.'

Label saved to /workspaces/tesis_v4/data/eval/dev/dev_103_20260717_161728.episode.json


## Status checkpoint

In [42]:
# Keep collecting dev seeds until five clean pilot trajectories are labelled.
!rwm eval status data/eval --manifest-path data/eval/seeds.json

=== Evaluation Data Status ===
  dev: 8 episodes
    quality: {'keep': 8}
  val: 0 episodes
    quality: {}
  locked_test: 0 episodes
    quality: {}

  Manifest: 24 seeds, hash=148a6cc22e6ab930
  No integrity issues detected.

  Total episodes: 8


## Development reward benchmark

In [43]:
# Select one primary episode per seed: the latest recording; duplicates remain available for diagnostics but do not double-weight a track.
import importlib
import rwm.evaluation.episode_evaluator as episode_evaluator
episode_evaluator = importlib.reload(episode_evaluator)  # Pick up evaluator changes without requiring a kernel restart.
from rwm.commands.rwm_manual_test import load_model
import numpy as np

# The active β=0.1 anchors include the corrected KL, connected Top-K, and vectorised training path.
ACTIVE_MODELS = {
    'beta=0.1, seed=42': ROOT / 'runs/component_refinement/02_vectorized_reward_anchor/beta0.1_seed42/checkpoint_best.pt',
    'beta=0.1, seed=43': ROOT / 'runs/component_refinement/02_vectorized_reward_anchor/beta0.1_seed43/checkpoint_best.pt',
}
missing = [str(path) for path in ACTIVE_MODELS.values() if not path.exists()]
assert not missing, 'Missing active checkpoint(s): ' + ', '.join(missing)
primary_by_seed = {}
for episode_path in sorted((EVAL_DIR / 'dev').glob('*.npz')):
    meta = episode_evaluator.assert_evaluation_episode_integrity(episode_path, MANIFEST, expected_split='dev')
    primary_by_seed[meta.track_seed] = episode_path
PRIMARY_EPISODES = [primary_by_seed[seed] for seed in sorted(primary_by_seed)]
print('Primary episodes:')
for path in PRIMARY_EPISODES:
    print(' -', path.name)

# This descriptive constant baseline is the mean reward over historical training rollouts, never over evaluation episodes.
TRAINING_ROLLOUTS = list((ROOT / 'data/rollouts/rwm_deterministic').rglob('*.npz'))
assert TRAINING_ROLLOUTS, 'Missing historical training rollouts.'
train_reward_mean = np.concatenate([np.load(path)['reward'] for path in TRAINING_ROLLOUTS]).mean()
print(f'Historical training reward mean: {train_reward_mean:.4f}')

Primary episodes:
 - dev_100_20260717_161029.npz
 - dev_101_20260717_161105.npz
 - dev_102_20260717_161215.npz
 - dev_103_20260717_161728.npz
 - dev_104_20260717_161800.npz
Historical training reward mean: 0.0780


In [44]:
# Score each transition once for each active anchor; this can take several minutes on the full development set.
import dataclasses
import pandas as pd

benchmarks = {}
for model_name, checkpoint in ACTIVE_MODELS.items():
    model = load_model(checkpoint, DEVICE)
    results = []
    for episode_path in PRIMARY_EPISODES:
        obs, actions, rewards, meta = episode_evaluator.load_evaluation_episode(episode_path)
        result = episode_evaluator.evaluate_episode(model, obs, actions, rewards, meta, train_reward_mean=train_reward_mean, device=str(DEVICE))
        results.append(dataclasses.asdict(result))
    benchmarks[model_name] = pd.DataFrame(results).sort_values('seed').reset_index(drop=True)

# Keep this compact per-episode table focused on the primary seed-42 anchor.
benchmark = benchmarks['beta=0.1, seed=42']
benchmark[['seed', 'steps', 'mse', 'mae', 'baseline_mse', 'cum_reward_true', 'cum_reward_pred', 'reward_positive_frac']]

/opt/conda/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


,seed,steps,mse,mae,baseline_mse,cum_reward_true,cum_reward_pred,reward_positive_frac
0,100,558,2.419176,1.161688,3.191725,462.718506,338.109924,0.249104
1,101,800,2.213628,1.312768,2.573290,603.168274,691.202026,0.257500
2,102,800,2.374308,1.234716,2.869734,615.340454,571.680176,0.241250
3,103,800,2.409945,1.313485,2.981948,632.230164,679.121887,0.245000
4,104,800,2.022027,1.171346,2.283370,490.469788,620.205994,0.207500


In [46]:
# Aggregate by transition count so a shorter episode does not receive the same weight as a full one.
def summarize(table):
    total_steps = table.steps.sum()
    summary = pd.Series({
        'episodes': len(table),
        'steps': total_steps,
        'weighted_mse': (table.mse * table.steps).sum() / total_steps,
        'weighted_mae': (table.mae * table.steps).sum() / total_steps,
        'weighted_baseline_mse': (table.baseline_mse * table.steps).sum() / total_steps,
    })
    summary['model_baseline_ratio'] = summary.weighted_mse / summary.weighted_baseline_mse
    return summary

model_summary = pd.DataFrame({name: summarize(table) for name, table in benchmarks.items()}).T

# Save metric tables only; evaluation data remains under data/eval and is never training input.
BENCHMARK_OUT = ROOT / 'runs/component_refinement/02_vectorized_reward_anchor/evaluation/development_human_v1'
BENCHMARK_OUT.mkdir(parents=True, exist_ok=True)
pd.concat([table.assign(model=name) for name, table in benchmarks.items()], ignore_index=True).to_csv(BENCHMARK_OUT / 'per_episode.csv', index=False)
model_summary.to_csv(BENCHMARK_OUT / 'model_summary.csv')
print(f'Saved: {BENCHMARK_OUT.relative_to(ROOT)}')

# Compare only these two independently seeded anchors on the same locked dev episodes; this is development evidence, not a test result.

Saved: runs/component_refinement/02_vectorized_reward_anchor/evaluation/development_human_v1


## Reward means by useful slice

In [47]:
# Mean true versus predicted reward reveals calibration by slice.
# Re-run the scoring cell above once after this notebook update so benchmark contains the slice summaries.
required_slice_columns = {'startup_n', 'steady_n', 'reward_event_steady_n', 'baseline_steady_n'}
missing_slice_columns = required_slice_columns - set(benchmark.columns)
assert not missing_slice_columns, f'Re-run the scoring cell first; missing: {sorted(missing_slice_columns)}'
slice_specs = [
    ('all', 'steps', 'cum_reward_true', 'cum_reward_pred'),
    ('startup <50', 'startup_n', 'startup_mean_true', 'startup_mean_pred'),
    ('steady', 'steady_n', 'steady_mean_true', 'steady_mean_pred'),
    ('reward events, steady', 'reward_event_steady_n', 'reward_event_steady_mean_true', 'reward_event_steady_mean_pred'),
    ('baseline -0.1, steady', 'baseline_steady_n', 'baseline_steady_mean_true', 'baseline_steady_mean_pred'),
]
rows = []
for _, episode in benchmark.iterrows():
    row = {('episode', ''): episode.episode_id, ('seed', ''): episode.seed}
    for label, n_key, true_key, pred_key in slice_specs:
        n = episode[n_key]
        row[(label, 'mean true')] = episode[true_key] / n if label == 'all' and n else episode[true_key]
        row[(label, 'mean pred')] = episode[pred_key] / n if label == 'all' and n else episode[pred_key]
    rows.append(row)

slice_matrix = pd.DataFrame(rows)
slice_matrix.columns = pd.MultiIndex.from_tuples(slice_matrix.columns)
display(slice_matrix.round(3))

episode seed       all           startup <50              steady  \
                         mean true mean pred   mean true mean pred mean true   
0  77a065efe731413b  100     0.829     0.606       0.048     0.597     0.906   
1  3b59b71ef0159d72  101     0.754     0.864       0.098     0.666     0.798   
2  ee3664aae02969bb  102     0.769     0.715       0.187     0.690     0.808   
3  18300faeb5dff527  103     0.790     0.849       0.116     0.658     0.835   
4  5ee068fc6a0336f7  104     0.613     0.775       0.034     0.675     0.652   

            reward events, steady           baseline -0.1, steady            
  mean pred             mean true mean pred             mean true mean pred  
0     0.607                 3.604     0.858                  -0.1     0.513  
1     0.877                 3.200     0.867                  -0.1     0.881  
2     0.716                 3.484     0.824                  -0.1     0.680  
3     0.862                 3.516     0.960                  -0.1     0.827  
4     0.782                 3.317     0.896                  -0.1     0.750

## Active-anchor comparison

In [ ]:
# Same locked development episodes, two independent training seeds; lower ratio is better.
display(model_summary[['episodes', 'steps', 'weighted_mse', 'weighted_mae', 'weighted_baseline_mse', 'model_baseline_ratio']].round(3))